# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to explore, load, and process the FAIR² Mass Spectrometry dataset.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. This is important for understanding the structure and selecting relevant data subsets for analysis.

In [ ]:
# List all available record sets and their fields
print("Record Sets and their Fields:\n")
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set @id: {record_set.id} (name: {record_set.name})")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id} (name: {field.name}, data type: {field.data_type})")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        * Column @id: {col.id} (name: {col.name}, data type: {col.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified in the overview step.

In [ ]:
# Example: Extract data from all available record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display column names for each record set
for record_set_id in record_set_ids:
    print(f"\nRecord Set: {record_set_id}")
    df = dataframes[record_set_id]
    print("Columns:", list(df.columns))
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. This helps prepare the data for downstream analysis.

In [ ]:
# Example EDA on the main protein record set
# Please adjust the selected record set and field IDs according to the printed overview above.
selected_record_set_id = record_set_ids[0]  # Assuming the primary record set is the first one listed
df = dataframes[selected_record_set_id]

# Choose a numeric field for analysis (e.g., 'Coverage','MW')
numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    threshold = df[numeric_field].quantile(0.75)  # Filter to top 25% values
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (top 25%):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field}:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a plausible categorical field
    group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print(f"No numeric fields found for record set {selected_record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This helps with understanding trends, patterns, and outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot the distribution of the chosen numeric field
if numeric_field_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have a group_field, make a boxplot
    if group_fields:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, you loaded FAIR² Mass Spectrometry data using `mlcroissant`, explored its structure by referencing Croissant `@id`s, performed basic data selection and EDA, and visualized numeric data distributions. You can now proceed to apply domain-specific analyses, further visualizations, or machine learning on the processed dataset.